### Objective: Develop a synthetic dataset generator
*Prepared by: Farras Ezra on 14-Jun 2026*

Use case: Football match score results of World Cup group stage. Consists of:
1. Match dates
2. Name of competing teams
3. Scores of each team

P.S. to avoid exceeding usage limit and time, number of matches is limited to 10

What should be tried: Quantization, Different shapes and sizes, use Gradio UI

What tools to use: pad token, eos token, apply chat template, etc.

## Step-by-step

1. Block 1: Environment setup and Model Loading
- Import Bitsandbytes, accelerate, transformers, gradio
- Huggingface login (HF Token)
- Define BitsAndBytesConfig (make it 4-bit and nf4)
- Pick models to start => Phi-4-mini-instruct, Llama-3.2-3B-Instruct, gemma-3-270m-it

2. Block 2: Prompt Engineering and Generation

Create a prompt and call apply_chat_template to generate following data:
- Match date
- Team A and team B (country)
- Score of team A and B

3. Block 3: Gradio UI
Build minimum Gradio interface with:
- Drop-down to pick a model (phi / lamma / gemma)
- "Generate Data" button that triggers the prompt
- Output text box that shows the raw data (in JSON) and the structured table (using dataframe)

### Block 1 - Environment and model initialization

In [ ]:
# Installing packages

!pip install -q gradio bitsandbytes accelerate transformers==4.57.6

MODELS = {
    "phi": "microsoft/Phi-4-mini-instruct",
    "llama": "meta-llama/Llama-3.2-3B-Instruct",
    "gemma": "google/gemma-3-270m-it"

}

In [ ]:
# Importing libraries

import torch
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

In [ ]:
# Logging in to HuggingFace

from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Checking Active GPU

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")

In [ ]:
# Quantization configuration

quant_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Block 2: Define Prompt and Inference Model

In [ ]:
# Wrap everything: Tokenizer, Causal LM, and Outputs, into one function: Generate(model, messages, quant, max_new_tokens)

def generate(model_name, messages, quant=True, max_new_tokens=800):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token # pad: Padding token, eos: end of sequence token
  input_ids = tokenizer.apply_chat_template(
      messages,
      return_tensors="pt",
      add_generation_prompt=True).to("cuda")
  streamer = TextStreamer(tokenizer)
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")

  if quant:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config
    ).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

  outputs = model.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=max_new_tokens,
      streamer=streamer)

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
## Sanity check only, skip this

# Define the prompt
# Define prompt

system_prompt = """You are a football match data generator.
Output ONLY a raw JSON array. No explanation, no markdown, no example."""

user_prompt = """Generate FIFA World Cup 2026 group stage matches with diverse teams.
Use countries from Africa, Asia, Europe, South America, North America.
Return ONLY a raw JSON array with keys: date, team_a, team_a_score, team_b, team_b_score.
Generate exactly 10 results"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Print the result
response = generate(MODELS["phi"], messages)

print(response)

In [ ]:
def parse_response(response):
    # Skip first array if it's the prompt example
    first_end = response.find("]") + 1
    json_start = response.find("[", first_end)

    # Fallback: if no second array, use the first one
    if json_start == -1:
        json_start = response.find("[")

    raw = response[json_start:]

    # Take whichever is further: last "}," or last "}"
    last_with_comma = raw.rfind("},")
    last_without_comma = raw.rfind("}")
    last_complete = max(last_with_comma + 1, last_without_comma)  # +1 to include the comma position

    clean = raw[:last_complete+1] + "]"
    parsed = json.loads(clean)
    return pd.json_normalize(parsed).head(10)

In [ ]:
## Sanity check only, skip this

parse_response(response)

### Block 3: Initiate Gradio Interface

In [ ]:
# Import Gradio

import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

In [ ]:
# Define the match data generator model

def generate_match_data(num_matches, model_choice):
    system_prompt = """You are a football match data generator.
Output ONLY a raw JSON array. No explanation, no markdown, no example."""

    user_prompt = f"""Generate FIFA World Cup 2026 group stage matches with diverse teams.
Use countries from Africa, Asia, Europe, South America, North America.
Return ONLY a raw JSON array with keys: date, team_a, team_a_score, team_b, team_b_score.
Generate exactly {num_matches} results"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    # Setup
    use_quant = model_choice != "gemma"
    tokenizer = AutoTokenizer.from_pretrained(MODELS[model_choice])
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True).to("cuda")

    if use_quant:
        model = AutoModelForCausalLM.from_pretrained(
            MODELS[model_choice],
            quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODELS[model_choice]).to("cuda")

    # Stream token by token
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,           # don't stream the input prompt back
        skip_special_tokens=True)   # don't stream <|end|> etc.

    thread = Thread(target=model.generate, kwargs=dict(
        input_ids=input_ids,
        max_new_tokens=1500,
        streamer=streamer))

    thread.start()

    # Yield tokens as they arrive to Raw Response box
    partial = ""
    for token in streamer:
        partial += token
        yield partial, pd.DataFrame()  # stream text, empty table for now

    # Once complete, parse and update table
    try:
        df = parse_response(partial)
        yield partial, df  # final yield with both text and table
    except Exception as e:
        yield f"Error parsing: {e}\n\n{partial}", pd.DataFrame()

In [ ]:
## Sanity check before initiating Gradio

for raw, df in generate_match_data(10, "phi"):
    pass  # keep overwriting until the last yield

# Now print the final result
print(raw[:])  # first 200 chars of raw response
print(df)         # final DataFrame

In [ ]:
# Initiate the Gradio UI

with gr.Blocks(title="World Cup Dataset Generator") as demo:
    gr.Markdown("## World Cup 2026 Hypothetical Match Data Generator")

    with gr.Row():
        num_matches = gr.Slider(
            minimum=5,
            maximum=20,
            value=10,
            step=1,
            label="Number of Matches")
        model_input = gr.Dropdown(
            choices=["phi", "llama", "gemma"],
            label="Select Model",
            value="phi")

    generate_btn = gr.Button("Generate Match Data", variant="primary")

    with gr.Row():
        raw_output = gr.Textbox(label="Raw Response", lines=10)
        table_output = gr.Dataframe(label="Parsed Dataset")

    generate_btn.click(
        fn=generate_match_data,
        inputs=[num_matches, model_input],
        outputs=[raw_output, table_output])

demo.launch(share=True)